# Krok 5b: model powrotu gracza w D1

**Pytanie:** które zachowania z pierwszego dnia gry wiążą się z powrotem następnego dnia (D1) i czy spadek retencji po 27.08.2018 znika, gdy uwzględnimy zachowanie graczy w dniu 0?

**Dane:** `data/step5a_player_features_d0.csv`, wynik zapytania `sql/step5a_player_features_d0.sql` (jeden wiersz na gracza z kohort, który mógł wrócić w D1).

**Etap 1 (ten notatnik na razie):** przygotowanie danych do regresji logistycznej. Model w kolejnym etapie.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)

## 1. Wczytanie tabeli cech z kroku 5a

Kolumny `True`/`False` z BigQuery pandas wczytuje od razu jako typ logiczny (`bool`).

In [2]:
features_raw = pd.read_csv(
    "data/step5a_player_features_d0.csv",
    parse_dates=["cohort_date"],
)

features_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 4239 entries, 0 to 4238
Data columns (total 16 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   user_pseudo_id                 4239 non-null   str           
 1   cohort_date                    4239 non-null   datetime64[us]
 2   cohort_period                  4239 non-null   str           
 3   platform                       4239 non-null   str           
 4   source_group                   4239 non-null   str           
 5   engagement_minutes_d0          4239 non-null   float64       
 6   quickplay_levels_started_d0    4239 non-null   int64         
 7   quickplay_levels_ended_d0      4239 non-null   int64         
 8   quickplay_levels_completed_d0  4239 non-null   int64         
 9   quickplay_win_rate_d0          1773 non-null   float64       
 10  played_level_mode_d0           4239 non-null   bool          
 11  leveled_up_d0               

**Kontrola:** liczba graczy i powrotów w obu okresach powinna zgadzać się z krokiem 4c (2 605 / 602 oraz 1 634 / 224).

In [3]:
(
    features_raw
    .groupby("cohort_period")
    .agg(
        players=("user_pseudo_id", "size"),
        returned_d1=("returned_d1", "sum"),
        d1_retention=("returned_d1", "mean"),
    )
    .round(3)
)

,players,returned_d1,d1_retention
cohort_period,,,
1_do_2018-08-26,2605,602,0.231
2_od_2018-08-27,1634,224,0.137


## 2. Przygotowanie danych

Trzy decyzje (uzasadnienie w dzienniku projektu):

1. **Braki w odsetku wygranych (58% graczy nie zakończyło żadnego poziomu quickplay):** nowa cecha tak/nie `finished_quickplay_level_d0`, a brakujący odsetek wygranych = 0. Model oddziela efekt „zagrał do końca poziomu” od efektu „jak mu szło”.
2. **Skośne liczby (czas gry, rozpoczęte poziomy):** `log(1 + x)` (`np.log1p`), żeby kilku graczy z setkami poziomów nie zdominowało modelu.
3. **Współliniowość:** z cech liczbowych zostają tylko czas gry i liczba rozpoczętych poziomów; zakończone i ukończone poziomy służyły tylko do odsetka wygranych.

Dodatkowo: okres kohorty jako cecha tak/nie `after_0827`, platforma i źródło jako kategorie, a cechy logiczne jako 0/1 (tego oczekuje model).

In [4]:
BINARY_COLUMNS = [
    "returned_d1",                   # cel modelu
    "finished_quickplay_level_d0",
    "played_level_mode_d0",
    "leveled_up_d0",
    "ran_out_of_extra_steps_d0",
    "used_extra_steps_d0",
    "had_error_d0",
    "after_0827",
]

MODEL_COLUMNS = [
    "user_pseudo_id",
    "log_engagement_minutes_d0",
    "log_quickplay_levels_started_d0",
    "quickplay_win_rate_d0",
    "platform",
    "source_group",
    *BINARY_COLUMNS,
]


def prepare_model_data(df: pd.DataFrame) -> pd.DataFrame:
    """Zamienia tabelę cech z kroku 5a w tabelę gotową do regresji logistycznej."""
    return (
        df
        .assign(
            # decyzja 1: wskaźnik braku danych + brakujący odsetek wygranych = 0
            finished_quickplay_level_d0=lambda d: d["quickplay_levels_ended_d0"].gt(0),
            quickplay_win_rate_d0=lambda d: d["quickplay_win_rate_d0"].fillna(0),
            # decyzja 2: log(1 + x) dla skośnych liczb
            log_engagement_minutes_d0=lambda d: np.log1p(d["engagement_minutes_d0"]),
            log_quickplay_levels_started_d0=lambda d: np.log1p(d["quickplay_levels_started_d0"]),
            # okres kohorty jako cecha tak/nie, platforma i źródło jako kategorie
            after_0827=lambda d: d["cohort_period"].eq("2_od_2018-08-27"),
            platform=lambda d: d["platform"].astype("category"),
            source_group=lambda d: d["source_group"].astype("category"),
        )
        # decyzja 3: tylko wybrane cechy
        .filter(MODEL_COLUMNS)
        .astype({column: "int8" for column in BINARY_COLUMNS})
    )


model_data = features_raw.pipe(prepare_model_data)

model_data.head()

,user_pseudo_id,log_engagement_minutes_d0,log_quickplay_levels_started_d0,quickplay_win_rate_d0,platform,source_group,returned_d1,finished_quickplay_level_d0,played_level_mode_d0,leveled_up_d0,ran_out_of_extra_steps_d0,used_extra_steps_d0,had_error_d0,after_0827
0,06609A3933F8675D7CAEEC4831E49CD1,0.641854,0.000000,0.0,ANDROID,google-play_organic,1,0,1,1,0,0,0,0
1,06DE385FB8FE1F9B3C866E5645866023,0.000000,0.000000,0.0,IOS,direct,1,0,0,0,0,0,0,0
2,08259AC19C7EC2FF98005884C341BCA7,0.285179,0.000000,0.0,ANDROID,direct,0,0,0,0,0,0,0,0
3,14820087732D43CA92C816CA22E9FDCD,2.420368,1.609438,1.0,ANDROID,direct,0,1,1,1,1,1,1,0
4,17DC2360AE6B8EA8BAA10074F9905BAF,0.636577,0.000000,0.0,IOS,direct,0,0,1,0,0,0,0,0


## 3. Kontrola gotowej tabeli

Po przygotowaniu nie może zostać żaden brak danych, a liczba wierszy musi się zgadzać (4 239).

In [5]:
print(f"Wiersze: {len(model_data)}, braki danych: {model_data.isna().sum().sum()}")

model_data.describe().T.round(2)

Wiersze: 4239, braki danych: 0


,count,mean,std,min,25%,50%,75%,max
log_engagement_minutes_d0,4239.0,1.47,1.10,0.0,0.58,1.29,2.19,5.61
log_quickplay_levels_started_d0,4239.0,1.00,1.00,0.0,0.00,0.69,1.61,5.41
quickplay_win_rate_d0,4239.0,0.16,0.31,0.0,0.00,0.00,0.12,1.00
returned_d1,4239.0,0.19,0.40,0.0,0.00,0.00,0.00,1.00
finished_quickplay_level_d0,4239.0,0.42,0.49,0.0,0.00,0.00,1.00,1.00
played_level_mode_d0,4239.0,0.47,0.50,0.0,0.00,0.00,1.00,1.00
leveled_up_d0,4239.0,0.43,0.50,0.0,0.00,0.00,1.00,1.00
ran_out_of_extra_steps_d0,4239.0,0.11,0.31,0.0,0.00,0.00,0.00,1.00
used_extra_steps_d0,4239.0,0.19,0.39,0.0,0.00,0.00,0.00,1.00
had_error_d0,4239.0,0.04,0.19,0.0,0.00,0.00,0.00,1.00


**Pierwsze spojrzenie przed modelem:** D1 w podziale na każdą cechę tak/nie. To porównanie „surowe”, bez uwzględnienia pozostałych cech; model w kolejnym etapie pokaże, które różnice zostają, gdy pozostałe cechy są stałe.

In [6]:
BINARY_FEATURES = [column for column in BINARY_COLUMNS if column != "returned_d1"]

(
    model_data
    .melt(
        id_vars="returned_d1",
        value_vars=BINARY_FEATURES,
        var_name="feature",
        value_name="value",
    )
    .groupby(["feature", "value"])
    .agg(
        players=("returned_d1", "size"),
        d1_retention=("returned_d1", "mean"),
    )
    .round(3)
)

players  d1_retention
feature                     value                       
after_0827                  0         2605         0.231
                            1         1634         0.137
finished_quickplay_level_d0 0         2466         0.163
                            1         1773         0.239
had_error_d0                0         4074         0.194
                            1          165         0.218
leveled_up_d0               0         2403         0.144
                            1         1836         0.261
played_level_mode_d0        0         2262         0.146
                            1         1977         0.250
ran_out_of_extra_steps_d0   0         3769         0.189
                            1          470         0.245
used_extra_steps_d0         0         3452         0.178
                            1          787         0.268

## 4. Modele regresji logistycznej

**Regresja logistyczna** szacuje, jak każda cecha zmienia szansę na powrót w D1, **przy pozostałych cechach stałych**.

Wyniki pokazuję jako **ilorazy szans** (odds ratio, OR):
- **OR = 1**: cecha nie zmienia szansy na powrót,
- **OR > 1**: cecha zwiększa szansę (np. OR = 1,5 to szansa większa o 50%),
- **OR < 1**: cecha zmniejsza szansę (np. OR = 0,6 to szansa mniejsza o 40%).

Przy każdym OR jest 95% przedział ufności (`ci_low`–`ci_high`). Jeśli przedział **zawiera 1**, efekt nie jest istotny. To samo mówi `p_value` > 0,05.

Dwa modele:
- **A (bazowy):** okres + platforma + źródło, czyli efekt okresu bez uwzględnienia zachowania,
- **B (pełny):** A + zachowanie w dniu 0.

Kategorie bazowe (z nimi porównywane są pozostałe): platforma **ANDROID**, źródło **direct**, okres **do 26.08**.

In [7]:
import statsmodels.formula.api as smf

FORMULA_BASE = "returned_d1 ~ after_0827 + C(platform) + C(source_group, Treatment('direct'))"

BEHAVIOR_FEATURES = [
    "log_engagement_minutes_d0",
    "log_quickplay_levels_started_d0",
    "finished_quickplay_level_d0",
    "quickplay_win_rate_d0",
    "played_level_mode_d0",
    "leveled_up_d0",
    "used_extra_steps_d0",
    "ran_out_of_extra_steps_d0",
    "had_error_d0",
]
FORMULA_FULL = FORMULA_BASE + " + " + " + ".join(BEHAVIOR_FEATURES)

model_base = smf.logit(FORMULA_BASE, data=model_data).fit(disp=False)
model_full = smf.logit(FORMULA_FULL, data=model_data).fit(disp=False)

print(FORMULA_FULL)

returned_d1 ~ after_0827 + C(platform) + C(source_group, Treatment('direct')) + log_engagement_minutes_d0 + log_quickplay_levels_started_d0 + finished_quickplay_level_d0 + quickplay_win_rate_d0 + played_level_mode_d0 + leveled_up_d0 + used_extra_steps_d0 + ran_out_of_extra_steps_d0 + had_error_d0


In [8]:
def odds_ratio_table(model) -> pd.DataFrame:
    """Zamienia współczynniki modelu na ilorazy szans z 95% przedziałem ufności i p-value."""
    return (
        pd.concat([model.params, model.conf_int()], axis=1)
        .set_axis(["odds_ratio", "ci_low", "ci_high"], axis=1)
        .pipe(np.exp)                       # współczynnik logitu -> iloraz szans
        .assign(p_value=model.pvalues)
        .drop(index="Intercept")
        .rename(index=lambda name: name
                .replace("C(source_group, Treatment('direct'))[T.", "source_group[")
                .replace("C(platform)[T.", "platform["))
        .round(3)
    )


odds_ratio_table(model_base)

,odds_ratio,ci_low,ci_high,p_value
platform[IOS],1.427,1.170,1.741,0.00
source_group[google-play_organic],1.850,1.528,2.240,0.00
source_group[inne],0.851,0.414,1.747,0.66
after_0827,0.542,0.457,0.642,0.00


In [9]:
odds_ratio_table(model_full)

,odds_ratio,ci_low,ci_high,p_value
platform[IOS],1.067,0.850,1.341,0.575
source_group[google-play_organic],1.187,0.963,1.464,0.108
source_group[inne],0.787,0.371,1.671,0.533
after_0827,0.545,0.456,0.651,0.000
log_engagement_minutes_d0,1.778,1.607,1.967,0.000
log_quickplay_levels_started_d0,1.127,1.003,1.267,0.045
finished_quickplay_level_d0,0.827,0.628,1.089,0.176
quickplay_win_rate_d0,1.028,0.750,1.410,0.863
played_level_mode_d0,0.747,0.421,1.325,0.318
leveled_up_d0,1.674,0.938,2.988,0.081


### Kluczowe porównanie: efekt okresu w modelu A i B

Jeśli OR dla `after_0827` w modelu B jest wyraźnie bliższy 1 niż w modelu A, spadek retencji wynika (częściowo) z innego zachowania graczy w dniu 0. Jeśli się nie zmienia, gracze po 27.08 zachowywali się w dniu 0 podobnie, a mimo to rzadziej wracali.

**Pseudo R²** (McFadden) mówi, jaką część „niepewności” co do powrotu wyjaśnia model (0 = nic, im więcej, tym lepiej; w danych o zachowaniu 0,1–0,2 to typowe wartości).

In [10]:
(
    pd.DataFrame({
        "model_A_bazowy": odds_ratio_table(model_base).loc["after_0827"],
        "model_B_pelny": odds_ratio_table(model_full).loc["after_0827"],
    })
    .T
    .assign(pseudo_r2=[round(model_base.prsquared, 3), round(model_full.prsquared, 3)])
)

,odds_ratio,ci_low,ci_high,p_value,pseudo_r2
model_A_bazowy,0.542,0.457,0.642,0.0,0.024
model_B_pelny,0.545,0.456,0.651,0.0,0.097


## 5. Czy gracze po 27.08 zachowywali się inaczej w dniu 0?

Opis bez modelu: typowe zachowanie w dniu 0 w obu okresach (na oryginalnej skali, nie po logarytmie).

In [11]:
(
    features_raw
    .groupby("cohort_period")
    .agg(
        median_engagement_minutes=("engagement_minutes_d0", "median"),
        median_quickplay_levels_started=("quickplay_levels_started_d0", "median"),
        finished_quickplay_level=("quickplay_levels_ended_d0", lambda s: s.gt(0).mean()),
        played_level_mode=("played_level_mode_d0", "mean"),
        leveled_up=("leveled_up_d0", "mean"),
        used_extra_steps=("used_extra_steps_d0", "mean"),
        ran_out_of_extra_steps=("ran_out_of_extra_steps_d0", "mean"),
        had_error=("had_error_d0", "mean"),
    )
    .T
    .round(3)
)

cohort_period,1_do_2018-08-26,2_od_2018-08-27
median_engagement_minutes,2.740,2.505
median_quickplay_levels_started,1.000,2.000
finished_quickplay_level,0.398,0.450
played_level_mode,0.488,0.433
leveled_up,0.456,0.397
used_extra_steps,0.165,0.219
ran_out_of_extra_steps,0.097,0.133
had_error,0.033,0.049


## 6. Ocena modelu: AUC (scikit-learn)

**AUC** (pole pod krzywą ROC) mówi, jak dobrze model **odróżnia** graczy, którzy wrócą, od tych, którzy nie wrócą:
- **0,5**: model nie odróżnia ich lepiej niż rzut monetą,
- **1,0**: model odróżnia ich bezbłędnie,
- interpretacja wprost: AUC = szansa, że losowy gracz, który wrócił, dostanie od modelu wyższą ocenę niż losowy gracz, który nie wrócił.

AUC liczę **na graczach, których model nie widział przy dopasowaniu** (walidacja krzyżowa, 5 części): model uczy się na 4/5 graczy, a ocenę liczę na pozostałej 1/5, i tak 5 razy. Dzięki temu wynik nie jest zawyżony przez „uczenie się na pamięć”.

`C=np.inf` wyłącza regularyzację, więc model w scikit-learn to ten sam model co w `statsmodels` (sprawdzenie w ostatniej komórce).

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

BASE_FEATURES = ["after_0827", "platform", "source_group"]


def design_matrix(df: pd.DataFrame, features: list) -> pd.DataFrame:
    """Wybiera cechy i zamienia kategorie na kolumny 0/1 (bez kategorii bazowej, jak w statsmodels)."""
    return pd.get_dummies(
        df.filter(features),
        columns=["platform", "source_group"],
        drop_first=True,        # ANDROID i direct jako kategorie bazowe
        dtype="int8",
    )


X_base = design_matrix(model_data, BASE_FEATURES)
X_full = design_matrix(model_data, BASE_FEATURES + BEHAVIOR_FEATURES)
y = model_data["returned_d1"]

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
logit_no_penalty = LogisticRegression(C=np.inf, max_iter=5000)

auc_by_model = {
    name: cross_val_score(logit_no_penalty, X, y, cv=folds, scoring="roc_auc")
    for name, X in [("model_A_bazowy", X_base), ("model_B_pelny", X_full)]
}

(
    pd.DataFrame(auc_by_model)
    .agg(["mean", "std", "min", "max"])
    .T
    .rename(columns={"mean": "AUC", "std": "AUC_std", "min": "AUC_min", "max": "AUC_max"})
    .round(3)
)

,AUC,AUC_std,AUC_min,AUC_max
model_A_bazowy,0.605,0.016,0.586,0.625
model_B_pelny,0.709,0.028,0.680,0.755


**Sprawdzenie zgodności:** ilorazy szans z scikit-learn (bez regularyzacji) i ze `statsmodels` powinny być praktycznie identyczne. Drobne różnice na trzecim miejscu po przecinku wynikają z innego algorytmu dopasowania.

In [13]:
sklearn_full = LogisticRegression(C=np.inf, max_iter=5000).fit(X_full, y)

(
    pd.DataFrame({
        "odds_ratio_sklearn": np.exp(sklearn_full.coef_[0]),
    }, index=X_full.columns)
    .rename(index=lambda name: name
            .replace("platform_", "platform[").replace("source_group_", "source_group[")
            + ("]" if name.startswith(("platform_", "source_group_")) else ""))
    .join(odds_ratio_table(model_full)[["odds_ratio"]].rename(columns={"odds_ratio": "odds_ratio_statsmodels"}))
    .round(3)
)

,odds_ratio_sklearn,odds_ratio_statsmodels
after_0827,0.544,0.545
log_engagement_minutes_d0,1.777,1.778
log_quickplay_levels_started_d0,1.127,1.127
finished_quickplay_level_d0,0.828,0.827
quickplay_win_rate_d0,1.025,1.028
played_level_mode_d0,0.748,0.747
leveled_up_d0,1.672,1.674
used_extra_steps_d0,1.025,1.030
ran_out_of_extra_steps_d0,0.820,0.814
had_error_d0,0.778,0.781


## 7. Zapis wyników modelu B (źródło wykresu w README)

In [14]:
(
    odds_ratio_table(model_full)
    .rename_axis("feature")
    .reset_index()
    .to_csv("data/step5b_odds_ratios_model_b.csv", index=False)
)